# 05 - Kafka → HDFS (Speed Layer → Archivage)
Consumer 2 : stocke les messages Kafka (Decembre) dans HDFS en Parquet

Ces donnees archivees sont disponibles pour un retraitement batch ulterieur

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# JARs Kafka montes dans /opt/spark-kafka-jars/
KAFKA_JARS = ','.join([
    '/opt/spark-kafka-jars/spark-sql-kafka-0-10_2.12-3.5.0.jar',
    '/opt/spark-kafka-jars/spark-token-provider-kafka-0-10_2.12-3.5.0.jar',
    '/opt/spark-kafka-jars/kafka-clients-3.4.1.jar',
    '/opt/spark-kafka-jars/commons-pool2-2.11.1.jar'
])

spark = SparkSession.builder \
    .appName('05_Kafka_to_HDFS') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.jars', KAFKA_JARS) \
    .getOrCreate()

print('Spark connecte')

Spark connecte


In [5]:
# Schema des messages Kafka (avec event_time du producer)
trip_schema = StructType([
    StructField('date_id', DoubleType()),
    StructField('pickup_location_id', IntegerType()),
    StructField('dropoff_location_id', IntegerType()),
    StructField('weather_id', DoubleType()),
    StructField('trip_distance', DoubleType()),
    StructField('total_amount', DoubleType()),
    StructField('event_time', StringType())
])

# Lire le flux Kafka
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka:29092') \
    .option('subscribe', 'nyc-taxi-trips') \
    .option('startingOffsets', 'earliest') \
    .load()

# Parser les messages JSON
trips_parsed = kafka_df \
    .select(from_json(col('value').cast('string'), trip_schema).alias('data')) \
    .select('data.*')

print('Flux Kafka connecte')

Flux Kafka connecte


In [ ]:
# Ecrire le flux dans HDFS en Parquet (micro-batches)
# Path dedie aux donnees streaming Decembre
HDFS_OUTPUT = 'hdfs://namenode:9000/projet/streaming/december_trips'
CHECKPOINT = 'hdfs://namenode:9000/projet/streaming/_checkpoint_december'

query = trips_parsed.writeStream \
    .outputMode('append') \
    .format('parquet') \
    .option('path', HDFS_OUTPUT) \
    .option('checkpointLocation', CHECKPOINT) \
    .trigger(processingTime='5 seconds') \
    .start()

print(f'Kafka → HDFS demarre (toutes les 5s)')
print(f'Les donnees Decembre arrivent dans : {HDFS_OUTPUT}')
print(f'Checkpoint : {CHECKPOINT}')

In [7]:
# Verifier les donnees ecrites dans HDFS
import time
time.sleep(15)  # Attendre quelques micro-batches

try:
    hdfs_trips = spark.read.parquet(HDFS_OUTPUT)
    print(f'Lignes stockees dans HDFS : {hdfs_trips.count()}')
    hdfs_trips.show(5)
except Exception as e:
    print(f'Pas encore de donnees : {e}')

Lignes stockees dans HDFS : 53675
+-------+------------------+-------------------+----------+-------------+------------+--------------------+
|date_id|pickup_location_id|dropoff_location_id|weather_id|trip_distance|total_amount|          event_time|
+-------+------------------+-------------------+----------+-------------+------------+--------------------+
|  346.0|               231|                100|      NULL|         1.24|        14.3|2026-05-28T02:37:...|
|  345.0|               159|                 48|      NULL|         0.95|       17.27|2026-05-28T02:37:...|
|  355.0|               162|                159|      NULL|         1.06|        14.0|2026-05-28T02:37:...|
|  366.0|               234|                227|      NULL|         1.93|       12.41|2026-05-28T02:37:...|
|  357.0|                90|                244|      NULL|         0.45|        10.1|2026-05-28T02:37:...|
+-------+------------------+-------------------+----------+-------------+------------+----------------

In [8]:
# Re-executez pour voir le compteur augmenter
hdfs_trips = spark.read.parquet(HDFS_OUTPUT)
print(f'Lignes stockees dans HDFS : {hdfs_trips.count()}')

Lignes stockees dans HDFS : 53675


In [9]:
# Arreter le consumer HDFS
query.stop()
print('Kafka → HDFS arrete')

# Stats finales
hdfs_trips = spark.read.parquet(HDFS_OUTPUT)
print(f'\nTotal stocke dans HDFS : {hdfs_trips.count()} lignes de Decembre')
print('Ces donnees sont archivees pour retraitement batch ulterieur')

spark.stop()
print('Spark session fermee')

Kafka → HDFS arrete

Total stocke dans HDFS : 53675 lignes de Decembre
Ces donnees sont archivees pour retraitement batch ulterieur
Spark session fermee
